In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
import tensorflow as tf
from tensorflow import keras
from keras.models import Model, Sequential
from keras.layers import Input, Conv1D, MaxPooling1D, Dense, Dropout, Flatten, Concatenate, LSTM, BatchNormalization, SpatialDropout1D, GlobalAveragePooling1D
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.losses import Huber
import os
import random
from ProfitStrategy import riskless_profit, profits_summary

In [2]:
folder = "exported_csvs"

data = {
    os.path.splitext(f)[0]: pd.read_csv(os.path.join(folder, f))
    for f in os.listdir(folder) if f.endswith(".csv")
}

In [3]:
train_idx, val_idx, test_idx = np.array(data['train_indices']).ravel(), np.array(data['val_indices']).ravel() ,np.array(data['test_indices']).ravel()

back_features = [
    #'back_prices_0', 'back_volumes_0', 'back_prices_1', 'back_volumes_1',
    #'back_VWAP', 
    #'others_back_PVT', 
    #'back_PVT',
    'WOM', 
    'back_VWAP_log_return'
]
lay_features = [
    #'lay_prices_0', 'lay_volumes_0', 'lay_prices_1', 'lay_volumes_1',
    #'lay_VWAP', 
    #'others_lay_PVT', 
    #'lay_PVT',
    'WOM', 
    'lay_VWAP_log_return'
]

back_target_feature = 'back_VWAP_log_return'
lay_target_feature = 'lay_VWAP_log_return'

commisions = 0.05

cur_back, cur_lay = np.array( data['back_VWAP'].iloc[test_idx, -2]), np.array( data['lay_VWAP'].iloc[test_idx, -2] )
future_back, future_lay = np.array( data['back_VWAP'].iloc[test_idx, -1] ), np.array( data['lay_VWAP'].iloc[test_idx, -1] )

perfect_riskless_profit = np.array([
    riskless_profit(cb, cl, fb, fl, fob, fol)
    for cb, cl, fb, fl, fob, fol in zip(
        cur_back, cur_lay,
        future_back, future_lay,
        future_back, future_lay
    )
])

def profit_summary(x):
    return profits_summary(x, perfect_riskless_profit)

In [4]:
SEED = 5

def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
def set_global_determinism(seed=SEED):
    set_seeds(seed=seed)

    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)

In [5]:
def Prepare_Data(selected_features, target_feature, dict_data = data, 
                 train_idx = train_idx, val_idx = val_idx, test_idx = test_idx):
    
    X = np.stack([dict_data[feat].iloc[:, :-1] for feat in selected_features], axis=-1)
    y = np.array(dict_data[target_feature].iloc[:, -1]).reshape(-1, 1)

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
    
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    for i in range(X_train.shape[-1]):
        X_train[:,:,i] = scaler_X.fit_transform(X_train[:,:,i])
        X_val[:,:,i] = scaler_X.transform(X_val[:,:,i])
        X_test[:,:,i] = scaler_X.transform(X_test[:,:,i])

    y_train = scaler_y.fit_transform(y_train).ravel()
    y_val = scaler_y.transform(y_val).ravel()

    return X_train, X_val, X_test, y_train, y_val, y_test.ravel(), scaler_y

def evaluate(y_test, y_pred):
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)*100
    rmse = np.sqrt(mse)
    correct_sign = sum( np.sign(y_test) == np.sign(y_pred) ) / len(y_test) * 100
    metrics = {
        #'MSE': mse,
        'RMSE': rmse,
        #'MAE': mae,
        'MAPE': mape,
        #'R2': r2,
        'correct_sign': correct_sign
    }
    return metrics

In [6]:
def CNN_LSTM_Model(input_shape=(9,len(back_features))):
    model = Sequential([
        Input(shape=input_shape),
        
        Conv1D(filters=64, kernel_size=3, 
               #padding='same', 
               activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        SpatialDropout1D(0.2),
        
        Conv1D(filters=32, kernel_size=3, 
               #padding='same', 
               activation='relu'),
        BatchNormalization(),
        SpatialDropout1D(0.2),
        
        LSTM(64, return_sequences=True),
        LSTM(32, return_sequences=False),
    
        #GlobalAveragePooling1D(),
        Dense(50, activation='relu'),
        Dropout(0.3),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    
    def soft_sign_loss(y_true, y_pred):
    # Loss = large when y_true and y_pred have opposite signs
    # Use: - y_true * y_pred — positive when signs match, negative when they don't
        return tf.reduce_mean(tf.nn.relu(-y_true * y_pred))


    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss= "mse"
        #loss = Huber(delta=1.0)
    )

    #model.summary()
    
    return model

In [7]:
def CNN_LSTM_Results(selected_features, target_feature, cur, dict_data = data, 
              train_idx = train_idx, test_idx = test_idx,
              epochs=100, batch_size=32, verbose=0):
    
    set_global_determinism(seed=SEED)
    
    model = CNN_LSTM_Model( input_shape = (9, len(selected_features) ) )

    X_train_scaled, X_val_scaled, X_test_scaled, y_train_scaled, y_val_scaled, y_test, scaler_y = \
    Prepare_Data(selected_features, target_feature)

    callbacks = [
            EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7)
        ]

    model.fit(X_train_scaled, y_train_scaled, epochs=epochs, batch_size=batch_size,
              #validation_split=0.2,
              validation_data=(X_val_scaled, y_val_scaled),
              callbacks=callbacks,verbose=verbose)

    y_pred = model.predict(X_test_scaled)
    y_pred = scaler_y.inverse_transform(y_pred).ravel()

    print( "log-returns: \n", evaluate(y_test, y_pred) )
    
    y_pred = cur*np.exp(y_pred.ravel())
    y_test = cur*np.exp(y_test.ravel())
    
    return {'forecast':y_pred, 'model':model, 'metrics': evaluate(y_test, y_pred)}

In [8]:
back_results = CNN_LSTM_Results(back_features, back_target_feature, cur_back)
lay_results = CNN_LSTM_Results(lay_features, lay_target_feature, cur_lay)
print( back_results['metrics'] )
print( lay_results['metrics'] )
forecast_back = np.array(back_results['forecast'])
forecast_lay = np.array(lay_results['forecast'])
CNN_LSTM_riskless_profit = np.array([
    riskless_profit(cb, cl, fb, fl, fob, fol)
    for cb, cl, fb, fl, fob, fol in zip(
        cur_back, cur_lay,
        future_back, future_lay,
        forecast_back, forecast_lay
    )
])
print( profit_summary(CNN_LSTM_riskless_profit) )

270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 994us/step
log-returns: 
 {'RMSE': 0.13526014048841936, 'MAPE': 301.4779487713348, 'correct_sign': 57.658909343522055}
270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step
log-returns: 
 {'RMSE': 0.13939217558199354, 'MAPE': 316.09567229222705, 'correct_sign': 54.532823897186525}
{'RMSE': 22.318883691181956, 'MAPE': 9.571493625149744, 'correct_sign': 100.0}
{'RMSE': 27.686286538001237, 'MAPE': 9.971616621800875, 'correct_sign': 100.0}
Min                                     -0.031262
1st Qu.                                 -0.014886
Median                                  -0.003317
Mean                                    -0.002454
3rd Qu.                                  0.006340
Max                                      0.026535
Betted Number                            9.000000
Betting Rate in Total Bets               0.001042
Betting Rate in Profitable Bets          0.003928
Profit Sum                              -0.022088
Profit Sum/Total Possible Profit       